# Modbus/TCP PCAP Feature Extraction for Anomaly Detection

This notebook extracts Modbus/TCP fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection.

In [51]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [52]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os
import numpy as np

In [53]:
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """
    Extract Modbus/TCP fields from a PCAP file using tshark.
    
    Args:
        pcap_path: Path to the PCAP file
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract (mbtcp = Modbus/TCP header, modbus = PDU)
    fields = [
        "frame.number",
        "frame.time_epoch",        
        "frame.len",                # Total frame length                   ANOMALOUS?
        "_ws.col.protocol",         # Protocols in frame
        "ip.src",
        "ip.dst",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",                  # TCP segment length                   ANOMALOUS?
        "tcp.stream",               # TCP flow identifier
        "tcp.time_delta",           # Time since previous packet
        "tcp.flags",                # TCP flags
        "tcp.seq",                  # Sequence number
        "tcp.ack",                  # Acknowledgment number
        "tcp.window_size",          # Window size
        "tcp.analysis.retransmission",      # Retransmission flag
        "tcp.analysis.duplicate_ack",       # Duplicate ACK flag
        "tcp.analysis.lost_segment",        # Lost segment flag
        "tcp.analysis.out_of_order",        # Out of order flag
        "tcp.analysis.flags",               # TCP analysis flags (e.g. [TCP Retransmission])
        "mbtcp.trans_id",           # Transaction Identifier               ANOMALOUS?
        "mbtcp.prot_id",
        "mbtcp.len",                # Length of remaining bytes in PDU     ANOMALOUS?
        "mbtcp.unit_id",            # Unit Identifier                      ANOMALOUS?
        "modbus.func_code",         # Function code                        ANOMALOUS?
        "modbus.reference_num",     # Reference number (address)           ANOMALOUS?
        "modbus.word_cnt",          # Number of data words in PDU          ANOMALOUS?
        "modbus.bit_cnt",           # Number of data bits in PDU
        "modbus.byte_cnt",          # Number of data bytes in PDU          ANOMALOUS?
        "modbus.exception_code",    # Exception code if present            ANOMALOUS?
        "modbus.request_frame",     # Request frame number
        "modbus.response_time",     # Response time
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", pcap_path,  
        "-T", "fields",
        "-Y", "modbus",  # Uncomment to filter only Modbus packets
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]

    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    # Process each line
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'protocols': parse_str(row.get('_ws.col.protocol')),
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'src_port': parse_int(row.get('tcp.srcport')),
            'dst_port': parse_int(row.get('tcp.dstport')),
            'tcp_len': parse_int(row.get('tcp.len')),
            'tcp_stream': parse_int(row.get('tcp.stream')),
            'tcp_time_delta': parse_float(row.get('tcp.time_delta')),
            'tcp_flags': parse_str(row.get('tcp.flags')),
            'tcp_seq': parse_int(row.get('tcp.seq')),
            'tcp_ack': parse_int(row.get('tcp.ack')),
            'tcp_window_size': parse_int(row.get('tcp.window_size')),
            'tcp_retransmission': 1 if parse_str(row.get('tcp.analysis.retransmission')) else 0,
            'tcp_duplicate_ack': 1 if parse_str(row.get('tcp.analysis.duplicate_ack')) else 0,
            'tcp_lost_segment': 1 if parse_str(row.get('tcp.analysis.lost_segment')) else 0,
            'tcp_out_of_order': 1 if parse_str(row.get('tcp.analysis.out_of_order')) else 0,
            'transaction_id': parse_int(row.get('mbtcp.trans_id')),
            'protocol_id': parse_int(row.get('mbtcp.prot_id')),
            'modbus_length': parse_int(row.get('mbtcp.len')),
            'unit_id': parse_int(row.get('mbtcp.unit_id')),
            'function_code': parse_int(row.get('modbus.func_code')),
            'reference_num': parse_int(row.get('modbus.reference_num')),
            'word_count': parse_int(row.get('modbus.word_cnt')),
            'bit_count': parse_int(row.get('modbus.bit_cnt')),
            'byte_count': parse_int(row.get('modbus.byte_cnt')),
            'exception_code': parse_int(row.get('modbus.exception_code')),
            'request_frame': parse_int(row.get('modbus.request_frame')),
            'response_frame': parse_int(row.get('modbus.response_frame')),
            'response_time': parse_float(row.get('modbus.response_time')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # Derive additional features
        record['is_request'] = 1 if record['dst_port'] == 502 else 0
        record['is_response'] = 1 if record['src_port'] == 502 else 0
        record['is_exception'] = 1 if record['exception_code'] is not None else 0
        
        records.append(record)
    
    return records

In [54]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names."""
    df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [55]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    import numpy as np
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Requests per second (rolling 1-second window) - optimized with searchsorted
    timestamps = df['timestamp'].values
    is_request = df['is_request'].values
    
    # Cumulative request count for efficient range queries
    cumulative_requests = np.cumsum(is_request)
    
    # For each packet, count requests in [timestamp - 1, timestamp]
    requests_per_second = np.zeros(len(df), dtype=int)
    for i, ts in enumerate(timestamps):
        # Find index of first packet within the 1-second window
        start_idx = np.searchsorted(timestamps, ts - 1, side='left')
        # Count requests from start_idx to i (inclusive)
        if start_idx > 0:
            requests_per_second[i] = cumulative_requests[i] - cumulative_requests[start_idx - 1]
        else:
            requests_per_second[i] = cumulative_requests[i]
    
    df['requests_per_second'] = requests_per_second
    
    return df

In [56]:
def parse_tcp_flags(df):
    """Parse TCP flags from hex string column in DataFrame."""
    
    def extract_flags(flags_hex):
        if not flags_hex or pd.isna(flags_hex):
            return pd.Series({
                'FIN': 0, 'SYN': 0, 'RST': 0, 
                'PSH': 0, 'ACK': 0, 'URG': 0
            })
        
        # Remove '0x' prefix if present
        flags_hex = str(flags_hex).replace('0x', '')
        try:
            flags = int(flags_hex, 16)
        except ValueError:
            return pd.Series({
                'FIN': 0, 'SYN': 0, 'RST': 0, 
                'PSH': 0, 'ACK': 0, 'URG': 0
            })
        
        return pd.Series({
            'FIN': 1 if flags & 0x001 else 0,
            'SYN': 1 if flags & 0x002 else 0,
            'RST': 1 if flags & 0x004 else 0,
            'PSH': 1 if flags & 0x008 else 0,
            'ACK': 1 if flags & 0x010 else 0,
            'URG': 1 if flags & 0x020 else 0,
        })
    
    # Apply to tcp_flags column and join back to df
    flag_cols = df['tcp_flags'].apply(extract_flags)
    df = pd.concat([df, flag_cols], axis=1)
    
    return df


In [57]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df


def add_rtt_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add round trip time (RTT) by matching Modbus requests with responses.
    
    RTT is calculated as the time between a request and its corresponding response,
    matched by transaction_id and IP address pairs.
    """
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['round_trip_time'] = np.nan
    
    # Dictionary to track pending requests: (transaction_id, client_ip, server_ip) -> (index, timestamp)
    pending_requests = {}
    
    for idx, row in df.iterrows():
        trans_id = row['transaction_id']
        
        if row['is_request'] == 1:
            # This is a request (going to port 502)
            # Key: (transaction_id, client_ip, server_ip)
            key = (trans_id, row['src_ip'], row['dst_ip'])
            pending_requests[key] = (idx, row['timestamp'])
            
        elif row['is_response'] == 1:
            # This is a response (coming from port 502)
            # Key: (transaction_id, client_ip, server_ip) - note src/dst are swapped
            key = (trans_id, row['dst_ip'], row['src_ip'])
            
            if key in pending_requests:
                req_idx, req_timestamp = pending_requests[key]
                rtt = row['timestamp'] - req_timestamp
                
                # Set RTT on both the response and the original request
                df.at[idx, 'round_trip_time'] = rtt
                df.at[req_idx, 'round_trip_time'] = rtt
                
                # Remove the matched request
                del pending_requests[key]
    
    return df

def add_flooding_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add features specific to query flooding detection."""
    
    # Sort by timestamp
    df = df.sort_values('timestamp')
    
    # Transaction ID patterns
    df['transaction_id_delta'] = df.groupby('flow_id')['transaction_id'].diff().fillna(0)
    df['transaction_id_repeats'] = df.groupby('flow_id')['transaction_id'].transform(
        lambda x: x.duplicated(keep=False).sum()
    )
    
    # Function code analysis
    df['func_code_entropy'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: -sum(p * np.log2(p) for p in x.value_counts(normalize=True) if p > 0)
    )
    df['consecutive_same_func_code'] = (
        df.groupby('flow_id')['function_code']
        .transform(lambda x: (x == x.shift()).cumsum())
    )
    
    # Request/Response ratios per flow
    df['flow_request_count'] = df.groupby('flow_id')['is_request'].transform('sum')
    df['flow_response_count'] = df.groupby('flow_id')['is_response'].transform('sum')
    df['response_rate'] = df['flow_response_count'] / df['flow_request_count'].replace(0, 1)
    
    # Exception rate
    df['flow_exception_count'] = df.groupby('flow_id')['is_exception'].transform('sum')
    df['exception_rate'] = df['flow_exception_count'] / df['flow_pkt_count'].replace(0, 1)
    
    # Rolling window features (for time-based flooding patterns)
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.set_index('timestamp_dt')
    
    df = df.reset_index(drop=True)
    
    return df

## Usage Example

In [58]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
benign_relative_path = "data\\raw\\Modbus Dataset\\benign\\ied1a\\ied1a-network-capture\\veth4edc015-normal-4.pcap"
attack_relative_path = "data\\raw\\Modbus Dataset\\attack\\external\\ied1a\\ied1a-network-capture\\veth4edc015-0.pcap"  # Example attack file
absolute_path = (Path("../..") / attack_relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = absolute_path  # Update this path
OUTPUT_CSV = "modbus_features.csv"

Using PCAP file at: C:\Users\jabrantes\OneDrive - Brookhaven National Laboratory\Documents\CODE STUFF\BNL\Foundational-Model-for-Energy-Security\data\raw\Modbus Dataset\attack\external\ied1a\ied1a-network-capture\veth4edc015-0.pcap


In [59]:
# Extract raw Modbus features
print(f"Processing: {PCAP_PATH}")
records = extract_modbus_features(PCAP_PATH)
print(f"Extracted {len(records)} Modbus packets")

Processing: C:\Users\jabrantes\OneDrive - Brookhaven National Laboratory\Documents\CODE STUFF\BNL\Foundational-Model-for-Energy-Security\data\raw\Modbus Dataset\attack\external\ied1a\ied1a-network-capture\veth4edc015-0.pcap
Extracted 137543 Modbus packets


In [60]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = parse_tcp_flags(df)
    df = add_temporal_features(df)
    # df = add_flow_features(df)
    df = add_rtt_feature(df)
    # df = add_flooding_features(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")


DataFrame shape: (137543, 47)

Columns: ['packet_number', 'timestamp', 'datetime', 'protocols', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'tcp_len', 'tcp_stream', 'tcp_time_delta', 'tcp_flags', 'tcp_seq', 'tcp_ack', 'tcp_window_size', 'tcp_retransmission', 'tcp_duplicate_ack', 'tcp_lost_segment', 'tcp_out_of_order', 'transaction_id', 'protocol_id', 'modbus_length', 'unit_id', 'function_code', 'reference_num', 'word_count', 'bit_count', 'byte_count', 'exception_code', 'request_frame', 'response_frame', 'response_time', 'pkt_len', 'is_request', 'is_response', 'is_exception', 'function_name', 'FIN', 'SYN', 'RST', 'PSH', 'ACK', 'URG', 'inter_arrival_time', 'time_from_start', 'requests_per_second', 'round_trip_time']


In [61]:
# Preview the data
df.head(10)

,packet_number,timestamp,datetime,protocols,src_ip,dst_ip,src_port,dst_port,tcp_len,tcp_stream,...,FIN,SYN,RST,PSH,ACK,URG,inter_arrival_time,time_from_start,requests_per_second,round_trip_time
0,5,1.675223e+09,2023-01-31T22:48:45.486440,Modbus/TCP,185.175.0.3,185.175.0.4,53454,502,12,1,...,0,0,0,1,1,0,NaN,0.000000,1,0.148968
1,15,1.675223e+09,2023-01-31T22:48:45.635408,Modbus/TCP,185.175.0.4,185.175.0.3,502,53454,11,1,...,0,0,0,1,1,0,0.148968,0.148968,1,0.148968
2,21,1.675223e+09,2023-01-31T22:48:45.656365,Modbus/TCP,185.175.0.3,185.175.0.4,53456,502,12,3,...,0,0,0,1,1,0,0.020957,0.169925,2,0.005697
3,26,1.675223e+09,2023-01-31T22:48:45.662062,Modbus/TCP,185.175.0.4,185.175.0.3,502,53456,11,3,...,0,0,0,1,1,0,0.005697,0.175622,2,0.005697
4,32,1.675223e+09,2023-01-31T22:48:45.663325,Modbus/TCP,185.175.0.3,185.175.0.4,53458,502,12,4,...,0,0,0,1,1,0,0.001263,0.176885,3,0.385779
5,39,1.675223e+09,2023-01-31T22:48:46.049104,Modbus/TCP,185.175.0.4,185.175.0.3,502,53458,12,4,...,0,0,0,1,1,0,0.385779,0.562664,3,0.385779
6,41,1.675223e+09,2023-01-31T22:48:46.049861,Modbus/TCP,185.175.0.3,185.175.0.4,53458,502,12,4,...,0,0,0,1,1,0,0.000757,0.563421,4,NaN
7,57,1.675223e+09,2023-01-31T22:49:21.085357,Modbus/TCP,185.175.0.3,185.175.0.4,53460,502,12,8,...,0,0,0,1,1,0,35.035496,35.598917,1,0.004211
8,62,1.675223e+09,2023-01-31T22:49:21.089568,Modbus/TCP,185.175.0.4,185.175.0.3,502,53460,11,8,...,0,0,0,1,1,0,0.004211,35.603128,1,0.004211
9,68,1.675223e+09,2023-01-31T22:49:21.110344,Modbus/TCP,185.175.0.3,185.175.0.4,53462,502,12,9,...,0,0,0,1,1,0,0.020776,35.623904,2,0.006084


In [62]:
# Summary statistics
print("Function Code Distribution:")
print(df['function_name'].value_counts())
print("\nBasic Statistics:")
df[['pkt_len', 'inter_arrival_time', 'modbus_length']].describe()

Function Code Distribution:
function_name
Write Single Coil         65640
Read Coils                28715
Read Holding Registers    14387
Read Input Registers      14379
Read Discrete Inputs      14376
Write Single Register        46
Name: count, dtype: int64

Basic Statistics:


,pkt_len,inter_arrival_time,modbus_length
count,137543.000000,137542.000000,137543.000000
mean,77.584363,0.267434,5.582443
std,1.031800,1.117474,0.745847
min,76.000000,0.000003,4.000000
25%,77.000000,0.001017,5.000000
50%,78.000000,0.001625,6.000000
75%,78.000000,0.020549,6.000000
max,342.000000,35.035496,6.000000


In [63]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")


Saved to: modbus_features.csv


## Feature Summary for Anomaly Detection

| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Protocol** | `function_code`, `unit_id`, `transaction_id` | Detect unauthorized commands |
| **Payload** | `reference_num`, `word_count`, `byte_count`, `reg_value` | Detect data manipulation |
| **Temporal** | `inter_arrival_time`, `rolling_iat_*`, `rtt` | Detect timing anomalies, DoS, network issues |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |
| **Error** | `is_exception`, `exception_code` | Detect probing, fuzzing |